# Base Place Recognition Pipeline 

Test Place Recognition on the 3DSSG dataset using `opr.pipelines`

In [1]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex, SequencePlaceRecognitionPipeline

from gsloc.inference.pr_infer import PRInferencer
from gsloc.models import opr_graph_extention as network
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-04-23 12:50:49.701 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


## Create dataset object

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Optional

import torch
from tqdm.auto import tqdm


def compute_scenegraph_edge_attr_mean_std(
    scenegraphs_root: str | Path,
    *,
    max_files: Optional[int] = None,
    log_every: int = 5000,
) -> dict[str, Any]:
    """Mean and std of ``edge_attr`` over all ``*.pt`` graphs under ``scenegraphs_root``.

    Each file may hold a ``torch_geometric.data.Data``, a ``dict`` with ``edge_attr``, or a
    nested ``list`` of those. Rows from every tensor are concatenated in the running sums.

    Args:
        scenegraphs_root: Root folder, e.g.
            ``/mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_real_classes_pt_compact``.
            If you use Windows with ``Y:`` mapped to this tree, pass
            ``Path(r"Y:\\mnt\\external_usb_hdd\\6YL\\Datasets\\3RScan\\SceneGraphs_real_classes_pt_compact")``.
        max_files: Stop after this many ``*.pt`` files (``None`` = all).
        log_every: Print progress every this many files.

    Returns:
        Dict with ``mean``, ``std`` (float32, shape ``[D]``), ``dim`` ``D``, ``num_edges``,
        ``num_files_used``, ``skipped`` (load failures + files with no usable ``edge_attr``).
    """
    from torch_geometric.data import Data

    root = Path(scenegraphs_root)
    if not root.is_dir():
        raise FileNotFoundError(f"Not a directory: {root}")

    files = sorted(root.rglob("*.pt"))
    if max_files is not None:
        files = files[: int(max_files)]

    sum_1d: Optional[torch.Tensor] = None
    sum_sq_1d: Optional[torch.Tensor] = None
    n_rows = 0
    dim: Optional[int] = None
    used_files = 0
    skipped = 0

    def _tensor_from(obj: Any) -> Optional[torch.Tensor]:
        if obj is None:
            return None
        if isinstance(obj, Data):
            ea = obj.edge_attr
        elif isinstance(obj, dict):
            ea = obj.get("edge_attr")
        else:
            return None
        if ea is None:
            return None
        ea = torch.as_tensor(ea, dtype=torch.float32) if not torch.is_tensor(ea) else ea.float()
        if ea.ndim == 1:
            ea = ea.unsqueeze(0)
        if ea.numel() == 0:
            return None
        return ea

    def _collect(obj: Any) -> list[torch.Tensor]:
        if obj is None:
            return []
        if isinstance(obj, list):
            out: list[torch.Tensor] = []
            for x in obj:
                out.extend(_collect(x))
            return out
        t = _tensor_from(obj)
        return [t] if t is not None else []

    for fi, path in enumerate(tqdm(files, desc="edge_attr mean/std")):
        if fi and log_every and fi % log_every == 0:
            print(f"  ... {fi:,} / {len(files):,} files, {n_rows:,} edge rows so far")

        try:
            loaded = torch.load(path, map_location="cpu", weights_only=False)
        except Exception:
            skipped += 1
            continue

        chunks = _collect(loaded)
        if not chunks:
            skipped += 1
            continue

        file_used = False
        for ea in chunks:
            d = ea.shape[1]
            if dim is None:
                dim = d
                sum_1d = torch.zeros(d, dtype=torch.float64)
                sum_sq_1d = torch.zeros(d, dtype=torch.float64)
            elif d != dim:
                skipped += 1
                continue

            ea64 = ea.to(dtype=torch.float64)
            sum_1d += ea64.sum(dim=0)
            sum_sq_1d += (ea64 * ea64).sum(dim=0)
            n_rows += ea64.shape[0]
            file_used = True

        if file_used:
            used_files += 1

    if dim is None or n_rows == 0:
        raise RuntimeError(f"No edge_attr rows found under {root}")

    mean64 = sum_1d / n_rows
    ex2 = sum_sq_1d / n_rows
    var = (ex2 - mean64 * mean64).clamp_min(0.0)
    mean = mean64.to(torch.float32)
    std = torch.sqrt(var).to(torch.float32)

    return {
        "mean": mean,
        "std": std,
        "dim": int(dim),
        "num_edges": int(n_rows),
        "num_files_used": int(used_files),
        "skipped": int(skipped),
        "root": str(root.resolve()),
    }


# Example (full scan can take a long time; use max_files=200 for a quick test):
_EDGE_STATS_ROOT = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_real_classes_pt_compact")
edge_stats = compute_scenegraph_edge_attr_mean_std(_EDGE_STATS_ROOT, max_files=50000)
edge_stats["mean"], edge_stats["std"], edge_stats["num_edges"], edge_stats["dim"]


edge_attr mean/std:   0%|          | 0/50000 [00:00<?, ?it/s]

  ... 5,000 / 50,000 files, 23,866 edge rows so far
  ... 10,000 / 50,000 files, 60,273 edge rows so far
  ... 15,000 / 50,000 files, 104,243 edge rows so far
  ... 20,000 / 50,000 files, 171,467 edge rows so far


KeyboardInterrupt: 

In [ ]:
from collections import Counter
from typing import Optional

import torch
from torch_geometric.data import Data


def edge_attr_feature_dim_from_pt(pt_path: str | Path) -> Optional[int]:
    """Return ``edge_attr.shape[1]`` for one ``.pt`` graph, or ``None`` if missing/empty."""
    path = Path(pt_path)
    obj = torch.load(path, map_location="cpu", weights_only=False)

    def _dim(ea) -> Optional[int]:
        if ea is None:
            return None
        t = torch.as_tensor(ea) if not torch.is_tensor(ea) else ea
        if t.numel() == 0:
            return None
        if t.ndim == 1:
            return 1
        return int(t.shape[1])

    if isinstance(obj, Data):
        return _dim(obj.edge_attr)
    if isinstance(obj, dict):
        return _dim(obj.get("edge_attr"))
    if isinstance(obj, list):
        dims = []
        for x in obj:
            if isinstance(x, Data):
                d = _dim(x.edge_attr)
            elif isinstance(x, dict):
                d = _dim(x.get("edge_attr"))
            else:
                d = None
            if d is not None:
                dims.append(d)
        if not dims:
            return None
        if len(set(dims)) != 1:
            raise ValueError(f"Inconsistent edge_attr widths in list inside {path}: {dims}")
        return dims[0]
    return None


def summarize_edge_attr_dims(
    root: str | Path,
    *,
    max_files: Optional[int] = None,
) -> dict[int, int]:
    """Count ``*.pt`` files under ``root`` by ``edge_attr`` feature dimension (``shape[1]``).

    Returns:
        ``Counter``-like mapping ``{D: num_files}`` for files with non-empty ``edge_attr``.
    """
    root = Path(root)
    files = sorted(root.rglob("*.pt"))
    if max_files is not None:
        files = files[: int(max_files)]

    counts: Counter = Counter()
    none_or_empty = 0
    errors = 0
    for p in files:
        try:
            d = edge_attr_feature_dim_from_pt(p)
        except Exception:
            errors += 1
            continue
        if d is None:
            none_or_empty += 1
        else:
            counts[d] += 1

    print("edge_attr feature dim (shape[1]) -> number of .pt files:")
    for dim in sorted(counts):
        print(f"  D={dim}: {counts[dim]}")
    print(f"  no / empty edge_attr: {none_or_empty}")
    print(f"  load/parse errors: {errors}")
    return dict(counts)


# Example:
summarize_edge_attr_dims("/mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_real_classes_pt_compact", max_files=500)
# edge_attr_feature_dim_from_pt("/path/to/single/frame-000000.pt")


edge_attr feature dim (shape[1]) -> number of .pt files:
  D=10: 377
  no / empty edge_attr: 123
  load/parse errors: 0


{10: 377}

In [2]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
test_dir = Path("/home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan")
index_path = test_dir / "index"
query_cache_path = test_dir / "query_cache"
bench_report_dir = test_dir / "seq_benchmark_report"

In [3]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [5]:
three_rscan_ds = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=index_path,
    rebuild_meta=True,  # meta.parquet already built
    # limit=20000,
    image_transform=image_transform_fn,
    save_meta=False,
    scene_filter_mode="listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
    graph_feat_dim = 4,
    graph_edge_attr_dim = 7,
    graph_rotate = True,
    graph_dir="SceneGraphs_real_classes_pt",
    # edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt",
)
# You can create your own dataloader for index generation
# dataloader = DataLoader(
#     three_rscan_ds, batch_size=16, shuffle=False, num_workers=4, collate_fn=three_rscan_ds.collate_fn
# )

2026-04-23 12:50:16.613 | INFO     | gsloc.datasets.three_rscan:__init__:290 - Rebuilding metadata for 3rscan dataset
2026-04-23 12:50:16.661 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:202 - Scanning 3rscan dataset for 30 selected scenes...
2026-04-23 12:50:26.150 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:237 - Scanned 9449 rows
2026-04-23 12:50:26.152 | WARNING  | gsloc.datasets.three_rscan:__init__:351 - Edge normalizer path is None; graph edge_attr will not be normalized


## Create model

In [6]:
# model = MegaLoc()
# model.eval()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

In [ ]:
# Load graph+MegaLoc head from ``gatv1/best_model.pth`` (hub MegaLoc uses different prefixes; conv stack is GAT in ckpt vs GINE in ``VPRGraphEncoder``).
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/2026-04-17_16-35-44/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

graph_enc = network.OPR_VPRGraphEncoder(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=528 + 1,
    num_edge_classes=41,
    node_emb_dim=128,
    edge_emb_dim=128,
    proj_dim=256,
)
model = network.OPR_MultiModalVPRGraphEncoder(
    graph_encoder=graph_enc,
    image_encoder=MegaLoc().model,
    image_out_dim=8448,
    graph_out_dim=256,
    fusion_dim=8448,
    normalize=True,
    graph_fusion_scale=0.05,
    freeze_image_encoder=True,
    mode="graph",
)
missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


AttributeError: 'MegaLoc' object has no attribute 'aggregator'

## Create Index (files that are used to do retrievel based on database)

In [56]:
# generate function runs model for all dataset's elements and generates 3 files that are need for retrievel
index = FaissFlatIndex.generate(
    directory=index_path,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model,
    rebuild_meta=True,
    rebuild_descriptors=True,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-04-23 12:04:17.297 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 9,449 rows to /home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan/index/meta.parquet
2026-04-23 12:04:17.298 | INFO     | mmpr.inference.index:generate:424 - meta.parquet file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan/index
100%|██████████| 394/394 [01:05<00:00,  6.04it/s]
2026-04-23 12:05:22.527 | INFO     | mmpr.inference.index:generate:453 - descriptors.npy file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan/index
2026-04-23 12:05:22.532 | INFO     | mmpr.inference.index:generate:473 - schema.json file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan/index


Index created at /home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan/index
Index size: 9449, dim: 256 metric: l2


# Test PlaceRecognitionPipeline

In [57]:
pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    # k=50,
)


seq_pr_pipeline = SequencePlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    max_window=25,
    per_frame_k=20,
    final_k=50,
    descriptor_agg="mean",
)

In [58]:
three_rscan_q = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=query_cache_path,
    # save_meta=True,
    rebuild_meta=False,
    # limit=10000,
    image_transform=image_transform_fn,
    scene_filter_mode="same_room_excluding_listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
    graph_feat_dim = 4,
    graph_edge_attr_dim = 10,
    graph_rotate = False,
    graph_dir="SceneGraphs_real_classes_pt_compact",
    # edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt",
)

2026-04-23 12:05:22.726 | WARNING  | gsloc.datasets.three_rscan:__init__:351 - Edge normalizer path is None; graph edge_attr will not be normalized


In [59]:
inferencer = PRInferencer(
    pr_pipeline=pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=25,
    device="cuda"
)

In [60]:
frames = inferencer.run(rebuild_query_descriptors=True)
inferencer.save(query_cache_path / "frames.npz", frames=frames)
# frames = inferencer.load(query_cache_path / "frames.npz")

Compute descriptors + PR cache: 100%|██████████| 1314/1314 [02:33<00:00,  8.56it/s]
2026-04-23 12:08:09.900 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/tests/gsloc/26-04-22/GraphEncoder/3rscan/query_cache/meta.parquet


In [14]:
frames2 = inferencer.load("/home/kartashov_ga/projects/tests/gsloc/26-04-22/MegaLoc/3rscan/query_cache/frames.npz")

In [49]:
from gsloc.inference.pr_infer import TwoModelsFrameMerge

itog_frames = TwoModelsFrameMerge(frames, frames2, distance_coef_a=10000000000, distance_coef_b=1)
inferencer.frames = itog_frames

In [52]:
inferencer.frames = frames

In [53]:
# inferencer.build_recall_benchmark_report(
#     database_dataset=three_rscan_ds,
#     ks=[1, 5, 10, 25],
#     similarity_kwargs={
#         "mode": "pose",
#         "trans_tol_m": 2,
#         "rot_tol_deg": 90
#         },
#     include_per_query=False
# )

In [61]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:10<00:00, 2041.67it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,6721,0.319850,31.984962
1,5,21013,10587,0.503831,50.383096
2,10,21013,12573,0.598344,59.834388
3,25,21013,15410,0.733356,73.335554


In [48]:
curr_report_dir = bench_report_dir / "room_k25_batch-std"

result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started


  0%|          | 0/11 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
result_df

,w,auc_pr,f1_max,recall_at_1,recall_at_1_std,recall_at_5,recall_at_5_std,recall_at_10,recall_at_10_std,recall_at_25,recall_at_25_std,num_valid,num_total
0,1,0.156056,0.283499,0.303336,0.044198,0.481273,0.048344,0.573930,0.048660,0.720792,0.043523,21013,21013
1,2,0.162454,0.290137,0.309190,0.047813,0.487270,0.052085,0.581354,0.050800,0.726503,0.044588,21013,21013
2,3,0.166558,0.292561,0.310141,0.047722,0.489554,0.052917,0.584638,0.051939,0.729691,0.045234,21013,21013
3,5,0.171962,0.293280,0.309665,0.049232,0.487888,0.052958,0.582592,0.054282,0.730215,0.044901,21013,21013
4,7,0.175576,0.292528,0.307048,0.046301,0.486794,0.051807,0.576881,0.051033,0.726645,0.047385,21013,21013
5,10,0.178989,0.292360,0.301432,0.047213,0.481797,0.051732,0.568886,0.050600,0.721982,0.046659,21013,21013
6,15,0.182928,0.290648,0.288869,0.046716,0.471327,0.050714,0.560701,0.052640,0.715414,0.045687,21013,21013
7,20,0.185020,0.289129,0.277590,0.045503,0.461000,0.048231,0.551611,0.047079,0.710941,0.043888,21013,21013
8,25,0.186488,0.289636,0.268643,0.048844,0.453767,0.050571,0.544139,0.048912,0.708419,0.046550,21013,21013
9,30,0.188222,0.290477,0.262266,0.044364,0.449388,0.052355,0.540856,0.054866,0.708371,0.049617,21013,21013


#Graph random report

In [18]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w', metrics, and optional '<metric>_std').
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = "all"

    for m in metrics:
        if m not in df.columns:
            continue

        std_col = f"{m}_std"
        has_std = std_col in df.columns

        line_kwargs = {
            "x": "w",
            "y": m,
            "title": f"{map_name}: {m} vs sequence length (w)",
            "markers": True,
        }
        if has_std:
            line_kwargs["error_y"] = std_col

        fig = px.line(df, **line_kwargs)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )

            weighted_mean_kwargs = {
                "x": wmean_series["w"],
                "y": wmean_series[m].astype(float),
                "mode": "lines",
                "name": "weighted mean",
                "line": dict(color="purple", dash="dot"),
                "showlegend": True,
            }

            if std_col in summary_all.columns:
                wstd_series = (
                    summary_all
                    .groupby("w")
                    .apply(lambda g: float(np.average(g[std_col].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                    .reset_index(name=std_col)
                )
                wmean_series = wmean_series.merge(wstd_series, on="w", how="left")
                weighted_mean_kwargs["error_y"] = dict(type="data", array=wmean_series[std_col].astype(float), visible=True)

            fig.add_trace(go.Scatter(**weighted_mean_kwargs))
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


### Base MegaLoc Results

In [30]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('26Ju6Xtn7j8DizLd/iPuP8Ean1dUAu' ... 'o5q+4/gj2mqUi07j+hnO2JOLruPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [35],
               'y': [0.9602320379170984]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [19]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('Lt30quMM0z+58UQe91fSPyLmOiqQNN' ... 'HyatQ/sM+0RBZ01D9eD5h854DUPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [35],
               'y': [0.3203676907289629]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [17]:
result_df_global_std = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=range(1, 25, 2),
    per_frame_k_used=10,
    save_dir=query_cache_path / "seq_pr_benchmark",
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/12 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  8%|▊         | 1/12 [00:37<06:54, 37.66s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 17%|█▋        | 2/12 [01:43<09:00, 54.01s/it]

...

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 92%|█████████▏| 11/12 [14:45<01:27, 87.95s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started


100%|██████████| 12/12 [16:14<00:00, 81.22s/it]

fused rankings preparation started


In [18]:
plot_metrics_vs_window_with_stats(result_df_global_std, result_df_global_std)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQMFBwkLDQ8RExUX', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('Xm2HB39n7j+FXLrAWALuP1In84QFEO' ... 'dNX2iY7j/omfCqu5/uPytsdnr5pe4/'),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [23],
               'y': [0.9577605621192381]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [17]:
seq_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
        },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:10<00:00, 1955.43it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,5938,0.282587,28.258697
1,5,21013,10549,0.502023,50.202256
2,10,21013,12317,0.586161,58.616095
3,25,21013,16555,0.787846,78.784562


In [ ]:
out = pipeline.infer(three_rscan_q[9000])

In [ ]:
out

PlaceRecognitionResult(descriptor=array([ 0.00143673,  0.00783881,  0.02143787, ...,  0.01978837,
       -0.00375643, -0.0045516 ], shape=(8448,), dtype=float32), indices=array([9000, 8787, 8786, 9001, 9005]), distances=array([8.8449399e-09, 5.0083816e-01, 7.0701253e-01, 7.5864244e-01,
       8.1422371e-01], dtype=float32), db_idx=array([9000, 8787, 8786, 9001, 9005]), db_pose=array([[ 0.798643  ,  0.983432  , -0.132033  ,  0.75250036, -0.5780175 ,
        -0.25381622, -0.18766014],
       [ 0.356377  ,  1.48875   , -0.131014  ,  0.8410546 , -0.427336  ,
        -0.3099954 , -0.11795727],
       [ 0.382755  ,  1.40634   , -0.0807452 ,  0.8403163 , -0.41481936,
        -0.32412416, -0.12937145],
       [ 0.841705  ,  0.995071  , -0.140846  ,  0.7504945 , -0.58146584,
        -0.24821363, -0.19247201],
       [ 0.860654  ,  0.989988  , -0.125784  ,  0.7427527 , -0.56332934,
        -0.28782725, -0.21939473]], dtype=float32))